
# Greedy model soup for PPO checkpoints

This notebook runs **Track 1**: greedy weight-space model soup.

It starts from one checkpoint, tries candidates one by one, and keeps a candidate **only if** the soup improves on the chosen evaluation metric.

Default behavior:

- start from `START_MODEL`
- try each checkpoint in `CANDIDATE_POOL`
- build a **trial soup** = average of current accepted members plus the candidate
- evaluate the trial soup on the configured suite
- **accept** the candidate if the metric improves
- continue until the soup is cooked and hopefully less idiotic than its ancestors

The notebook is self-contained around your current PPO/C4 stack and does not depend on the earlier distillation notebook.


In [1]:

import copy
import json
import math
import random
import re
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from C4.CNet192 import save_cnet192
from C4.connect4_env import Connect4Env
from C4.fast_connect4_lookahead import Connect4Lookahead
from PPO.actor_critic import ActorCritic, TransferCfg, NEG_INF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda



## Config

Edit this cell first.

A good default starting point for your current HOF is:

- start with one strong generalist
- try a compact pool of 4 to 5 similarly strong generalists
- let greedy soup decide which ones actually play well together

You can keep `SORT_POOL_BY_BASELINE = False` if you want to respect the order you type in.
Set it to `True` if you want the notebook to pre-score the pool and try stronger standalone candidates first.


In [2]:

# -------------------------------
# Main soup config
# -------------------------------

START_MODEL = "PPO_Models/PPO_817.pt"

CANDIDATE_POOL = [
    "PPO_Models/PPO_848.pt",
    "PPO_Models/PPO_827.pt", #good, always in, try without
    "PPO_Models/PPO_846.pt",
    "PPO_Models/PPO_832.pt",
    "PPO_Models/PPO_836.pt",
    "PPO_Models/PPO_838.pt",
    "PPO_Models/PPO_845.pt",
    "PPO_Models/PPO_834.pt",
    "PPO_Models/PPO_812.pt",
    "PPO_Models/PPO_842.pt",
    "PPO_Models/PPO_820.pt",
    "PPO_Models/SOUP_4.pt",
    "PPO_Models/SOUP_5.pt",
    "PPO_Models/PPO_915.pt",
]

# Optional: if True, evaluate standalone candidates first and sort pool by standalone score.
SORT_POOL_BY_BASELINE = True

# Which metric decides acceptance?
# Options created by this notebook:
#   "GS_CUSTOM"   weighted suite score
#   "AVG_SCORE"   simple mean across suite opponents
#   "LA_HARD"     mean of hard tactical opponents only (LA-3, LA-5, LA-6, LA-7, LA-9, LA-11, LA-13 if present)
METRIC_TO_MAXIMIZE = "GS_CUSTOM"

# Minimum improvement required to accept a candidate.
# Set to 0.0 for pure greedy. Small positive value avoids accepting noise.
MIN_IMPROVEMENT = 0.0

# Deterministic argmax for model actions during evaluation.
MODEL_DETERMINISTIC = True

# Reproducibility
SEED = 12345

# Output name for the final soup checkpoint.
SOUP_TAG = "PPO_SOUP"

# Save result artifacts
SAVE_SOUP_CHECKPOINT = True
SAVE_RESULTS_XLSX = True
SAVE_RESULTS_JSON = True

# -------------------------------
# Fast suite used during greedy selection
# Keep this reasonably small, because it is run many times.
# -------------------------------

FAST_EVAL_OPPONENTS = OrderedDict({
    "Random":   {"type": "random",    "games": 60},
    "Leftmost": {"type": "leftmost",  "games": 20},
    "Center":   {"type": "center",    "games": 40},
    "LA-1":     {"type": "lookahead", "depth": 1,  "games": 20},
    "LA-2":     {"type": "lookahead", "depth": 2,  "games": 20},
    "LA-3":     {"type": "lookahead", "depth": 3,  "games": 20},
    "LA-4":     {"type": "lookahead", "depth": 4,  "games": 10},
    "LA-5":     {"type": "lookahead", "depth": 5,  "games": 10},
    "LA-6":     {"type": "lookahead", "depth": 6,  "games": 10},
    "LA-7":     {"type": "lookahead", "depth": 7,  "games": 8},
    "LA-9":     {"type": "lookahead", "depth": 9,  "games": 6},
    "LA-11":    {"type": "lookahead", "depth": 11, "games": 4},
    "LA-13":    {"type": "lookahead", "depth": 13, "games": 4},
})

# -------------------------------
# Optional final suite
# Usually same as your standard suite, or equal to fast suite if you just want a quick run.
# -------------------------------

FINAL_EVAL_OPPONENTS = OrderedDict({
    "Random":   {"type": "random",    "games": 200},
    "Leftmost": {"type": "leftmost",  "games": 100},
    "Center":   {"type": "center",    "games": 200},
    "LA-1":     {"type": "lookahead", "depth": 1,  "games": 100},
    "LA-2":     {"type": "lookahead", "depth": 2,  "games": 100},
    "LA-3":     {"type": "lookahead", "depth": 3,  "games": 100},
    "LA-4":     {"type": "lookahead", "depth": 4,  "games": 30},
    "LA-5":     {"type": "lookahead", "depth": 5,  "games": 20},
    "LA-6":     {"type": "lookahead", "depth": 6,  "games": 12},
    "LA-7":     {"type": "lookahead", "depth": 7,  "games": 10},
    "LA-9":     {"type": "lookahead", "depth": 9,  "games": 6},
    "LA-11":    {"type": "lookahead", "depth": 11, "games": 4},
    "LA-13":    {"type": "lookahead", "depth": 13, "games": 4},
})


In [3]:

# -------------------------------
# Helpers: reproducibility, paths, env
# -------------------------------

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def make_env() -> Connect4Env:
    return Connect4Env()

CENTER_ORDER = [3, 4, 2, 5, 1, 6, 0]

def center_tiebreak(indices: List[int]) -> int:
    idx_set = set(int(i) for i in indices)
    for c in CENTER_ORDER:
        if c in idx_set:
            return c
    return int(sorted(indices)[0])

def ensure_state_tensor(state: np.ndarray, device: torch.device) -> torch.Tensor:
    x = torch.as_tensor(state, dtype=torch.float32, device=device)
    if x.dim() == 2:
        x = x.unsqueeze(0).unsqueeze(0)   # (1,1,6,7)
    elif x.dim() == 3:
        x = x.unsqueeze(0)                # (1,C,6,7)
    elif x.dim() != 4:
        raise ValueError(f"Unexpected state shape: {tuple(x.shape)}")
    return x

def state_to_board_pov(state: np.ndarray) -> np.ndarray:
    s = np.asarray(state)
    if s.ndim == 4:
        if s.shape[0] != 1:
            raise ValueError(f"Expected batch size 1, got {s.shape}")
        s = s[0]
    if s.ndim == 3:
        if s.shape[0] == 1:
            s = s[0]
        else:
            raise ValueError(f"Expected single-channel POV board, got {s.shape}")
    if s.shape != (6, 7):
        raise ValueError(f"Expected board shape (6,7), got {s.shape}")
    return s.astype(np.int8, copy=False)


In [4]:
# -------------------------------
# Checkpoint loading and soup utilities
# -------------------------------

def load_actor_critic_from_ckpt(path: str | Path, device: torch.device) -> ActorCritic:
    path = Path(path)
    ac = ActorCritic.from_cnet192_checkpoint(
        path=str(path),
        device=device,
        transfer=TransferCfg(
            strict_load=True,
            freeze_conv=False,
        ),
    )
    ac.eval()
    for p in ac.parameters():
        p.requires_grad_(False)
    return ac

def get_net_state_dict(ac: ActorCritic) -> OrderedDict:
    return OrderedDict((k, v.detach().cpu().clone()) for k, v in ac.net.state_dict().items())

def assert_compatible_state_dicts(reference: OrderedDict, other: OrderedDict, label: str = "") -> None:
    ref_keys = list(reference.keys())
    oth_keys = list(other.keys())
    if ref_keys != oth_keys:
        raise ValueError(f"Incompatible keys for {label}.")
    for k in ref_keys:
        if reference[k].shape != other[k].shape:
            raise ValueError(f"Incompatible tensor shape for key '{k}' in {label}: {reference[k].shape} vs {other[k].shape}")

def average_state_dicts(
    state_dicts: List[OrderedDict],
    weights: Optional[List[float]] = None,
    show_progress: bool = False,
    tqdm_desc: str = "Averaging state_dicts",
    tqdm_position: int = 2,
    tqdm_leave: bool = False,
) -> OrderedDict:
    if len(state_dicts) == 0:
        raise ValueError("average_state_dicts needs at least one state dict.")

    if weights is None:
        weights = [1.0 / len(state_dicts)] * len(state_dicts)
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
        weights = weights.tolist()

    ref = state_dicts[0]
    for i, sd in enumerate(state_dicts[1:], start=1):
        assert_compatible_state_dicts(ref, sd, label=f"state_dict[{i}]")

    out = OrderedDict()
    keys = list(ref.keys())

    key_iter = keys
    if show_progress:
        key_iter = tqdm(
            keys,
            desc=tqdm_desc,
            position=tqdm_position,
            leave=tqdm_leave,
        )

    for k in key_iter:
        t0 = ref[k]

        if torch.is_floating_point(t0):
            acc = None
            for w, sd in zip(weights, state_dicts):
                term = sd[k].float() * float(w)
                acc = term if acc is None else acc + term
            out[k] = acc.to(dtype=t0.dtype)
        else:
            # Non-floating buffers should match across compatible checkpoints.
            same = True
            for sd in state_dicts[1:]:
                if not torch.equal(t0, sd[k]):
                    same = False
                    break
            if not same:
                raise ValueError(f"Non-floating tensor mismatch for key '{k}'.")
            out[k] = t0.clone()

        if show_progress:
            shape_str = "x".join(str(x) for x in t0.shape) if hasattr(t0, "shape") else "scalar"
            key_iter.set_postfix({
                "key": k.split(".")[-1],
                "shape": shape_str,
            })

    return out

def build_soup_model(
    members: List[ActorCritic],
    member_paths: List[str],
    show_progress: bool = False,
    tqdm_position: int = 2,
) -> ActorCritic:
    if len(members) == 0:
        raise ValueError("Soup must contain at least one member.")

    if len(members) == 1:
        soup = copy.deepcopy(members[0])
        soup.eval()
        for p in soup.parameters():
            p.requires_grad_(False)
        soup._soup_member_paths = list(member_paths)
        return soup

    soup = copy.deepcopy(members[0])
    avg_sd = average_state_dicts(
        [get_net_state_dict(m) for m in members],
        show_progress=show_progress,
        tqdm_desc=f"Averaging {len(members)} models",
        tqdm_position=tqdm_position,
        tqdm_leave=False,
    )
    soup.net.load_state_dict(avg_sd, strict=True)
    soup.eval()
    for p in soup.parameters():
        p.requires_grad_(False)
    soup._soup_member_paths = list(member_paths)
    return soup

def pretty_name(path: str | Path) -> str:
    return Path(path).stem

In [5]:

# -------------------------------
# Opponent policies
# -------------------------------

class RandomOpponent:
    def __init__(self, seed: int = 0):
        self.rng = np.random.default_rng(seed)

    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return int(self.rng.choice(legal_actions))

class LeftmostOpponent:
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return int(min(legal_actions))

class CenterOpponent:
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return center_tiebreak(list(legal_actions))

class LookaheadOpponent:
    def __init__(self, depth: int):
        self.depth = int(depth)
        self.la = Connect4Lookahead()
        self.la.OPENING_RANDOM = False

    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        if len(legal_actions) == 1:
            return int(legal_actions[0])

        board = state_to_board_pov(state)
        scores = np.asarray(self.la.n_step_action_scores(board, player=1, depth=self.depth), dtype=np.float64)

        mask = np.zeros(7, dtype=bool)
        mask[legal_actions] = True
        scores[~mask] = -1e18

        best = np.max(scores[mask])
        best_cols = [c for c in legal_actions if abs(scores[c] - best) <= 1e-12]
        return center_tiebreak(best_cols)

def make_opponent(cfg: Dict, seed: int = 0):
    t = cfg["type"]
    if t == "random":
        return RandomOpponent(seed=seed)
    if t == "leftmost":
        return LeftmostOpponent()
    if t == "center":
        return CenterOpponent()
    if t == "lookahead":
        return LookaheadOpponent(depth=int(cfg["depth"]))
    raise ValueError(f"Unknown opponent type: {t}")

@torch.no_grad()
def choose_model_action(model: ActorCritic, state: np.ndarray, legal_actions: List[int], deterministic: bool = True) -> int:
    x = ensure_state_tensor(state, device=device)
    logits, _ = model(x)

    legal_mask = torch.zeros_like(logits, dtype=torch.bool)
    legal_mask[0, legal_actions] = True
    masked_logits = logits.masked_fill(~legal_mask, NEG_INF)

    if deterministic:
        vals = masked_logits[0].detach().cpu().numpy()
        best = np.max(vals[legal_actions])
        best_cols = [c for c in legal_actions if abs(vals[c] - best) <= 1e-12]
        return center_tiebreak(best_cols)

    probs = torch.softmax(masked_logits, dim=-1)[0].detach().cpu().numpy()
    probs = probs / probs.sum()
    return int(np.random.choice(np.arange(7), p=probs))


In [6]:
# -------------------------------
# Evaluation
# -------------------------------

def play_one_game(model: ActorCritic, opponent, model_starts: bool, seed: int) -> int:
    '''
    Returns:
        +1 if model wins
         0 if draw
        -1 if model loses
    '''
    env = make_env()
    state = env.reset()
    model_turn = bool(model_starts)

    for _ in range(42):
        legal = env.available_actions()
        if not legal:
            return 0

        if model_turn:
            action = choose_model_action(model, state, legal, deterministic=MODEL_DETERMINISTIC)
        else:
            action = int(opponent.choose(state, legal))

        prev_model_turn = model_turn
        state, reward, done = env.step(action)

        if done:
            # Assumes reward is from the mover perspective, which matches your PPO notebooks.
            if reward > 0:
                return +1 if prev_model_turn else -1
            return 0

        model_turn = not model_turn

    return 0

def opponent_weight(label: str, base: float = 1.4) -> float:
    if label == "Random":
        return 1.0
    if label in ("Leftmost", "Center"):
        return 1.0
    m = re.search(r"(\d+)", label)
    if m is None:
        return 1.0
    depth = int(m.group(1))
    return float(base ** depth)

def summarize_suite_row(row: Dict, suite: Dict[str, Dict]) -> Dict:
    score_cols = []
    hard_cols = []

    for label in suite.keys():
        if label in row:
            score_cols.append(label)
            if label in {"LA-3", "LA-5", "LA-6", "LA-7", "LA-9", "LA-11", "LA-13"}:
                hard_cols.append(label)

    avg_score = float(np.mean([row[c] for c in score_cols])) if score_cols else 0.0
    la_hard = float(np.mean([row[c] for c in hard_cols])) if hard_cols else 0.0

    weights = np.array([opponent_weight(c) for c in score_cols], dtype=np.float64)
    vals = np.array([row[c] for c in score_cols], dtype=np.float64)
    gs_custom = float((weights * vals).sum() / weights.sum()) if len(score_cols) else 0.0

    row["AVG_SCORE"] = avg_score
    row["LA_HARD"] = la_hard
    row["GS_CUSTOM"] = gs_custom
    return row

def evaluate_model_on_suite(
    model: ActorCritic,
    suite: Dict[str, Dict],
    model_name: str,
    seed: int = 0,
    show_progress: bool = True,
    tqdm_position: int = 0,
    tqdm_leave: bool = False,
) -> Tuple[pd.DataFrame, Dict]:
    row = OrderedDict()
    row["MODEL"] = model_name

    all_details = {}
    iterable = suite.items()

    if show_progress:
        iterable = tqdm(
            list(iterable),
            desc=f"Evaluating {model_name}",
            leave=tqdm_leave,
            position=tqdm_position,
        )

    for idx, (label, cfg) in enumerate(iterable):
        games = int(cfg["games"])
        opponent = make_opponent(cfg, seed=seed + 1000 * (idx + 1))

        wins = 0
        losses = 0
        draws = 0

        for g in range(games):
            model_starts = (g % 2 == 0)
            result = play_one_game(
                model=model,
                opponent=opponent,
                model_starts=model_starts,
                seed=seed + idx * 10000 + g,
            )
            if result > 0:
                wins += 1
            elif result < 0:
                losses += 1
            else:
                draws += 1

        score = (wins + 0.5 * draws) / games if games > 0 else 0.0
        row[label] = float(score)

        all_details[label] = {
            "wins": wins,
            "losses": losses,
            "draws": draws,
            "games": games,
            "score": float(score),
        }

        if show_progress:
            iterable.set_postfix({
                "opp": label,
                "score": f"{score:.3f}",
                "W-L-D": f"{wins}-{losses}-{draws}",
            })

    row = summarize_suite_row(row, suite)
    df = pd.DataFrame([row])
    return df, all_details

def display_eval(df: pd.DataFrame, metric: str = METRIC_TO_MAXIMIZE):
    score_cols = [c for c in df.columns if c.startswith("LA-") or c in {"Random", "Leftmost", "Center"}]
    extra_cols = ["AVG_SCORE", "LA_HARD", "GS_CUSTOM"]
    cols = ["MODEL"] + score_cols + [c for c in extra_cols if c in df.columns]
    show = df[cols].copy()
    for c in show.columns:
        if c != "MODEL":
            show[c] = show[c].map(lambda x: f"{float(x):.3f}")
    display(show)
    print(f"Metric [{metric}] =", float(df.iloc[0][metric]))

# -------------------------------
# FAST eval cache
# -------------------------------

FAST_EVAL_CACHE_PATH = Path("soup_fast_eval_cache.json")

def _suite_signature(suite: Dict[str, Dict]) -> str:
    serializable = []
    for label, cfg in suite.items():
        serializable.append((label, dict(cfg)))
    return json.dumps(serializable, sort_keys=True)

def _fast_eval_cache_key(
    model_path: str,
    suite: Dict[str, Dict],
    seed: int,
    deterministic: bool,
) -> str:
    payload = {
        "model_path": str(model_path),
        "suite": _suite_signature(suite),
        "seed": int(seed),
        "deterministic": bool(deterministic),
    }
    return json.dumps(payload, sort_keys=True)

def _jsonify_value(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    if isinstance(v, np.ndarray):
        return v.tolist()
    return v

def _normalize_record(d: Dict) -> Dict:
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = _normalize_record(v)
        elif isinstance(v, list):
            out[k] = [_jsonify_value(x) for x in v]
        else:
            out[k] = _jsonify_value(v)
    return out

def load_fast_eval_cache() -> Dict:
    if FAST_EVAL_CACHE_PATH.exists():
        try:
            return json.loads(FAST_EVAL_CACHE_PATH.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"Warning: failed to load FAST eval cache: {e}")
    return {}

def save_fast_eval_cache(cache: Dict) -> None:
    try:
        FAST_EVAL_CACHE_PATH.write_text(
            json.dumps(cache, indent=2),
            encoding="utf-8",
        )
    except Exception as e:
        print(f"Warning: failed to save FAST eval cache: {e}")

FAST_EVAL_CACHE = load_fast_eval_cache()

def get_fast_eval_cached(
    model,
    model_path: str,
    suite: Dict[str, Dict],
    model_name: str,
    seed: int,
    show_progress: bool = True,
    tqdm_position: int = 0,
    tqdm_leave: bool = False,
    save_cache: bool = True,
):
    key = _fast_eval_cache_key(
        model_path=model_path,
        suite=suite,
        seed=seed,
        deterministic=MODEL_DETERMINISTIC,
    )

    if key in FAST_EVAL_CACHE:
        cached = FAST_EVAL_CACHE[key]
        df = pd.DataFrame([cached["row"]])
        details = cached["details"]
        print(f"[cache hit] {model_name}")
        return df, details

    print(f"[cache miss] {model_name}")
    df, details = evaluate_model_on_suite(
        model=model,
        suite=suite,
        model_name=model_name,
        seed=seed,
        show_progress=show_progress,
        tqdm_position=tqdm_position,
        tqdm_leave=tqdm_leave,
    )

    FAST_EVAL_CACHE[key] = {
        "row": _normalize_record(df.iloc[0].to_dict()),
        "details": _normalize_record(details),
    }

    if save_cache:
        save_fast_eval_cache(FAST_EVAL_CACHE)

    return df, details

In [7]:

# -------------------------------
# Optional: preload and validate checkpoint pool
# -------------------------------

all_paths = [START_MODEL] + [p for p in CANDIDATE_POOL if p != START_MODEL]
print("Loading checkpoints...")

loaded_models: Dict[str, ActorCritic] = {}
failed_paths = {}

for p in all_paths:
    try:
        loaded_models[p] = load_actor_critic_from_ckpt(p, device=device)
        print("Loaded:", p)
    except Exception as e:
        failed_paths[p] = str(e)
        print("FAILED:", p)
        print("   ", e)

if START_MODEL not in loaded_models:
    raise RuntimeError("START_MODEL failed to load, soup cannot start.")

ref_sd = get_net_state_dict(loaded_models[START_MODEL])
compatible_pool = []
incompatible_pool = []

for p in CANDIDATE_POOL:
    if p not in loaded_models:
        incompatible_pool.append((p, "load failed"))
        continue
    try:
        assert_compatible_state_dicts(ref_sd, get_net_state_dict(loaded_models[p]), label=p)
        compatible_pool.append(p)
    except Exception as e:
        incompatible_pool.append((p, str(e)))

print("\nCompatible pool:")
for p in compatible_pool:
    print("  ", p)

if incompatible_pool:
    print("\nExcluded / incompatible candidates:")
    for p, msg in incompatible_pool:
        print("  ", p, "->", msg)

CANDIDATE_POOL = compatible_pool


Loading checkpoints...
Loaded: PPO_Models/PPO_817.pt
Loaded: PPO_Models/PPO_848.pt
Loaded: PPO_Models/PPO_827.pt
Loaded: PPO_Models/PPO_846.pt
Loaded: PPO_Models/PPO_832.pt
Loaded: PPO_Models/PPO_836.pt
Loaded: PPO_Models/PPO_838.pt
Loaded: PPO_Models/PPO_845.pt
Loaded: PPO_Models/PPO_834.pt
Loaded: PPO_Models/PPO_812.pt
Loaded: PPO_Models/PPO_842.pt
Loaded: PPO_Models/PPO_820.pt
Loaded: PPO_Models/SOUP_4.pt
Loaded: PPO_Models/SOUP_5.pt
Loaded: PPO_Models/PPO_915.pt

Compatible pool:
   PPO_Models/PPO_848.pt
   PPO_Models/PPO_827.pt
   PPO_Models/PPO_846.pt
   PPO_Models/PPO_832.pt
   PPO_Models/PPO_836.pt
   PPO_Models/PPO_838.pt
   PPO_Models/PPO_845.pt
   PPO_Models/PPO_834.pt
   PPO_Models/PPO_812.pt
   PPO_Models/PPO_842.pt
   PPO_Models/PPO_820.pt
   PPO_Models/SOUP_4.pt
   PPO_Models/SOUP_5.pt
   PPO_Models/PPO_915.pt



## Optional standalone prescore

If enabled, this scores each standalone candidate on the **fast suite** and sorts the trial order by that score.

This is often useful because greedy soup usually behaves better when stronger standalone models are tried earlier.


In [8]:
%%time

print("Preparing baseline prescore...")
print(f"Start model: {pretty_name(START_MODEL)}")
print(f"Candidates in pool: {len(CANDIDATE_POOL)}")
print(f"SORT_POOL_BY_BASELINE = {SORT_POOL_BY_BASELINE}")

baseline_rows = []

print("\nRunning FAST baseline for start model...")
start_df_fast, start_details_fast = get_fast_eval_cached(
    model=loaded_models[START_MODEL],
    model_path=START_MODEL,
    suite=FAST_EVAL_OPPONENTS,
    model_name=pretty_name(START_MODEL),
    seed=SEED,
    show_progress=True,
    tqdm_position=0,
    tqdm_leave=False,
)
display_eval(start_df_fast)

if SORT_POOL_BY_BASELINE and len(CANDIDATE_POOL) > 0:
    print("\nScoring standalone candidates on FAST_EVAL_OPPONENTS...")

    outer_bar = tqdm(
        list(enumerate(CANDIDATE_POOL, start=1)),
        desc="Standalone candidate prescore",
        total=len(CANDIDATE_POOL),
        position=0,
        leave=True,
    )

    for idx, p in outer_bar:
        model_name = pretty_name(p)

        outer_bar.set_postfix({
            "candidate": model_name,
        })

        print("\n" + "=" * 88)
        print(f"Prescore step {idx}: evaluating {model_name}")

        df_i, _ = get_fast_eval_cached(
            model=loaded_models[p],
            model_path=p,
            suite=FAST_EVAL_OPPONENTS,
            model_name=model_name,
            seed=SEED + 1 + idx,
            show_progress=True,
            tqdm_position=1,
            tqdm_leave=False,
        )

        row = df_i.iloc[0].to_dict()
        row["PATH"] = p
        baseline_rows.append(row)

        metric_val = float(row[METRIC_TO_MAXIMIZE])
        outer_bar.set_postfix({
            "candidate": model_name,
            METRIC_TO_MAXIMIZE: f"{metric_val:.4f}",
        })

        print(f"Finished {model_name} | {METRIC_TO_MAXIMIZE}: {metric_val:.6f}")

    baseline_df = pd.DataFrame(baseline_rows).sort_values(
        by=METRIC_TO_MAXIMIZE, ascending=False
    ).reset_index(drop=True)

    show_cols = ["MODEL", METRIC_TO_MAXIMIZE, "AVG_SCORE", "LA_HARD", "GS_CUSTOM"]
    show_cols = [c for c in show_cols if c in baseline_df.columns]
    display(baseline_df[show_cols])

    CANDIDATE_POOL = baseline_df["PATH"].tolist()

    print("\nPool order after standalone prescore:")
    for p in CANDIDATE_POOL:
        print("  ", p)
else:
    print("\nPool order will be used exactly as written in CANDIDATE_POOL.")

Preparing baseline prescore...
Start model: PPO_817
Candidates in pool: 14
SORT_POOL_BY_BASELINE = True

Running FAST baseline for start model...
[cache miss] PPO_817


Evaluating PPO_817:   0%|          | 0/13 [00:00<?, ?it/s]

,MODEL,Random,Leftmost,Center,LA-1,LA-2,LA-3,LA-4,LA-5,LA-6,LA-7,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_817,1.000,1.000,1.000,1.000,0.500,1.000,1.000,1.000,0.500,1.000,0.500,1.000,0.500,0.846,0.786,0.690


Metric [GS_CUSTOM] = 0.6904808288136184

Scoring standalone candidates on FAST_EVAL_OPPONENTS...


Standalone candidate prescore:   0%|          | 0/14 [00:00<?, ?it/s]


Prescore step 1: evaluating PPO_848
[cache hit] PPO_848
Finished PPO_848 | GS_CUSTOM: 0.637418

Prescore step 2: evaluating PPO_827
[cache hit] PPO_827
Finished PPO_827 | GS_CUSTOM: 0.300561

Prescore step 3: evaluating PPO_846
[cache hit] PPO_846
Finished PPO_846 | GS_CUSTOM: 0.493371

Prescore step 4: evaluating PPO_832
[cache miss] PPO_832


Evaluating PPO_832:   0%|          | 0/13 [00:00<?, ?it/s]

Finished PPO_832 | GS_CUSTOM: 0.481377

Prescore step 5: evaluating PPO_836
[cache hit] PPO_836
Finished PPO_836 | GS_CUSTOM: 0.298663

Prescore step 6: evaluating PPO_838
[cache hit] PPO_838
Finished PPO_838 | GS_CUSTOM: 0.493371

Prescore step 7: evaluating PPO_845
[cache hit] PPO_845
Finished PPO_845 | GS_CUSTOM: 0.304296

Prescore step 8: evaluating PPO_834
[cache hit] PPO_834
Finished PPO_834 | GS_CUSTOM: 0.565180

Prescore step 9: evaluating PPO_812
[cache hit] PPO_812
Finished PPO_812 | GS_CUSTOM: 0.498910

Prescore step 10: evaluating PPO_842
[cache hit] PPO_842
Finished PPO_842 | GS_CUSTOM: 0.521456

Prescore step 11: evaluating PPO_820
[cache hit] PPO_820
Finished PPO_820 | GS_CUSTOM: 0.554757

Prescore step 12: evaluating SOUP_4
[cache hit] SOUP_4
Finished SOUP_4 | GS_CUSTOM: 0.674085

Prescore step 13: evaluating SOUP_5
[cache hit] SOUP_5
Finished SOUP_5 | GS_CUSTOM: 0.674085

Prescore step 14: evaluating PPO_915
[cache hit] PPO_915
Finished PPO_915 | GS_CUSTOM: 0.928128


,MODEL,GS_CUSTOM,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_915,0.928128,0.882051,0.857143,0.928128
1,SOUP_4,0.674085,0.769231,0.785714,0.674085
2,SOUP_5,0.674085,0.769231,0.785714,0.674085
3,PPO_848,0.637418,0.689744,0.642857,0.637418
4,PPO_834,0.565180,0.769231,0.714286,0.565180
5,PPO_820,0.554757,0.769231,0.642857,0.554757
6,PPO_842,0.521456,0.652564,0.571429,0.521456
7,PPO_812,0.498910,0.653846,0.500000,0.498910
8,PPO_846,0.493371,0.615385,0.500000,0.493371
9,PPO_838,0.493371,0.615385,0.500000,0.493371



Pool order after standalone prescore:
   PPO_Models/PPO_915.pt
   PPO_Models/SOUP_4.pt
   PPO_Models/SOUP_5.pt
   PPO_Models/PPO_848.pt
   PPO_Models/PPO_834.pt
   PPO_Models/PPO_820.pt
   PPO_Models/PPO_842.pt
   PPO_Models/PPO_812.pt
   PPO_Models/PPO_846.pt
   PPO_Models/PPO_838.pt
   PPO_Models/PPO_832.pt
   PPO_Models/PPO_845.pt
   PPO_Models/PPO_827.pt
   PPO_Models/PPO_836.pt
CPU times: total: 3min 22s
Wall time: 3min 24s



## Greedy soup run

This is the main cell.

Logic:

- current soup starts as `START_MODEL`
- for each candidate:
  - build trial soup = average(current members + candidate)
  - evaluate trial soup on the fast suite
  - if chosen metric improves by at least `MIN_IMPROVEMENT`, accept it
  - otherwise reject it
- continue until all candidates are tested


In [9]:
%%time

print("Initializing greedy soup run...")
print(f"Start model: {pretty_name(START_MODEL)}")
print(f"Candidates in pool: {len(CANDIDATE_POOL)}")

history_rows = []
accepted_paths = [START_MODEL]
accepted_models = [loaded_models[START_MODEL]]

print("Preparing initial soup from START_MODEL...")
current_soup = copy.deepcopy(loaded_models[START_MODEL])
current_soup.eval()
for p in current_soup.parameters():
    p.requires_grad_(False)

print("Reusing cached start baseline as initial soup evaluation...")
current_fast_df = start_df_fast.copy()
current_fast_details = start_details_fast
current_metric = float(current_fast_df.iloc[0][METRIC_TO_MAXIMIZE])

history_rows.append({
    "step": 0,
    "candidate": pretty_name(START_MODEL),
    "accepted": True,
    "reason": "initial soup",
    "members": " + ".join(pretty_name(p) for p in accepted_paths),
    "metric_before": np.nan,
    "metric_trial": current_metric,
    "metric_after": current_metric,
})

print("\nInitial soup members:", [pretty_name(p) for p in accepted_paths])
print(f"Initial {METRIC_TO_MAXIMIZE}: {current_metric:.6f}")

outer_bar = tqdm(
    list(enumerate(CANDIDATE_POOL, start=1)),
    desc="Greedy soup candidates",
    total=len(CANDIDATE_POOL),
    position=0,
    leave=True,
)

for step_idx, candidate_path in outer_bar:
    candidate_name = pretty_name(candidate_path)
    metric_before = current_metric

    outer_bar.set_postfix({
        "candidate": candidate_name,
        "current": f"{current_metric:.4f}",
        "members": len(accepted_paths),
    })

    print("\n" + "=" * 88)
    print(f"Step {step_idx}: trying candidate {candidate_name}")
    print(f"Building trial soup with {len(accepted_paths) + 1} members...")

    trial_paths = accepted_paths + [candidate_path]
    trial_models = accepted_models + [loaded_models[candidate_path]]
    trial_soup = build_soup_model(
        trial_models,
        trial_paths,
        show_progress=True,
        tqdm_position=2,
    )

    print(f"Evaluating trial soup for {candidate_name}...")
    trial_df, trial_details = evaluate_model_on_suite(
        model=trial_soup,
        suite=FAST_EVAL_OPPONENTS,
        model_name="TRIAL_" + candidate_name,
        seed=SEED + step_idx,
        show_progress=True,
        tqdm_position=1,
        tqdm_leave=False,
    )

    trial_metric = float(trial_df.iloc[0][METRIC_TO_MAXIMIZE])
    delta = trial_metric - metric_before
    accepted = bool(delta >= MIN_IMPROVEMENT)

    if accepted:
        accepted_paths = trial_paths
        accepted_models = trial_models
        current_soup = trial_soup
        current_fast_df = trial_df
        current_fast_details = trial_details
        current_metric = trial_metric
        reason = f"accepted, Δ={delta:.6f}"
        print(f"ACCEPTED {candidate_name} | {METRIC_TO_MAXIMIZE}: {trial_metric:.6f} | delta={delta:.6f}")
    else:
        reason = f"rejected, Δ={delta:.6f}"
        print(f"REJECTED {candidate_name} | trial={trial_metric:.6f} | current={metric_before:.6f} | delta={delta:.6f}")

    history_rows.append({
        "step": step_idx,
        "candidate": candidate_name,
        "accepted": accepted,
        "reason": reason,
        "members": " + ".join(pretty_name(p) for p in accepted_paths),
        "metric_before": metric_before,
        "metric_trial": trial_metric,
        "metric_after": current_metric,
    })

    outer_bar.set_postfix({
        "candidate": candidate_name,
        "trial": f"{trial_metric:.4f}",
        "current": f"{current_metric:.4f}",
        "accepted": accepted,
        "members": len(accepted_paths),
    })

history_df = pd.DataFrame(history_rows)
display(history_df)

print("\nFinal accepted soup members:")
for p in accepted_paths:
    print("  ", pretty_name(p))

print(f"\nFinal FAST {METRIC_TO_MAXIMIZE}: {current_metric:.6f}")
display_eval(current_fast_df)

Initializing greedy soup run...
Start model: PPO_817
Candidates in pool: 14
Preparing initial soup from START_MODEL...
Reusing cached start baseline as initial soup evaluation...

Initial soup members: ['PPO_817']
Initial GS_CUSTOM: 0.690481


Greedy soup candidates:   0%|          | 0/14 [00:00<?, ?it/s]


Step 1: trying candidate PPO_915
Building trial soup with 2 members...


Averaging 2 models:   0%|          | 0/16 [00:00<?, ?it/s]

Evaluating trial soup for PPO_915...


...

REJECTED PPO_836 | trial=0.664425 | current=0.690481 | delta=-0.026056


,step,candidate,accepted,reason,members,metric_before,metric_trial,metric_after
0,0,PPO_817,True,initial soup,PPO_817,NaN,0.690481,0.690481
1,1,PPO_915,False,"rejected, Δ=-0.683039",PPO_817,0.690481,0.007442,0.690481
2,2,SOUP_4,False,"rejected, Δ=-0.010857",PPO_817,0.690481,0.679624,0.690481
3,3,SOUP_5,False,"rejected, Δ=-0.125395",PPO_817,0.690481,0.565086,0.690481
4,4,PPO_848,False,"rejected, Δ=-0.016396",PPO_817,0.690481,0.674085,0.690481
5,5,PPO_834,False,"rejected, Δ=-0.010857",PPO_817,0.690481,0.679624,0.690481
6,6,PPO_820,False,"rejected, Δ=-0.120078",PPO_817,0.690481,0.570403,0.690481
7,7,PPO_842,False,"rejected, Δ=-0.165162",PPO_817,0.690481,0.525319,0.690481
8,8,PPO_812,False,"rejected, Δ=-0.108905",PPO_817,0.690481,0.581576,0.690481
9,9,PPO_846,False,"rejected, Δ=-0.119983",PPO_817,0.690481,0.570497,0.690481



Final accepted soup members:
   PPO_817

Final FAST GS_CUSTOM: 0.690481


,MODEL,Random,Leftmost,Center,LA-1,LA-2,LA-3,LA-4,LA-5,LA-6,LA-7,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_817,1.000,1.000,1.000,1.000,0.500,1.000,1.000,1.000,0.500,1.000,0.500,1.000,0.500,0.846,0.786,0.690


Metric [GS_CUSTOM] = 0.6904808288136184
CPU times: total: 22min 15s
Wall time: 22min 28s



## Final evaluation on the larger suite


In [10]:

final_name = SOUP_TAG + "__" + "__".join(pretty_name(p) for p in accepted_paths)

final_df, final_details = evaluate_model_on_suite(
    model=current_soup,
    suite=FINAL_EVAL_OPPONENTS,
    model_name=final_name,
    seed=SEED + 999,
    show_progress=True,
)

display_eval(final_df)


Evaluating PPO_SOUP__PPO_817:   0%|          | 0/13 [00:00<?, ?it/s]

,MODEL,Random,Leftmost,Center,LA-1,LA-2,LA-3,LA-4,LA-5,LA-6,LA-7,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GS_CUSTOM
0,PPO_SOUP__PPO_817,1.000,1.000,1.000,1.000,0.500,1.000,1.000,1.000,0.500,1.000,0.500,1.000,0.500,0.846,0.786,0.690


Metric [GS_CUSTOM] = 0.6904808288136184


In [11]:

# -------------------------------
# Save outputs
# -------------------------------

DIR = "SOUP"

tag_suffix = "__".join(pretty_name(p) for p in accepted_paths)
safe_tag = f"{SOUP_TAG}__{tag_suffix}"

artifacts = {}

if SAVE_SOUP_CHECKPOINT:
    out_ckpt = Path("PPO_Models") / f"{safe_tag}.pt"
    out_ckpt.parent.mkdir(parents=True, exist_ok=True)

    save_cnet192(
        model=current_soup.net,
        path=out_ckpt,
        tag=safe_tag,
        session=safe_tag,
        episode=0,
        seed=SEED,
        t=0,
        cfg_override={"use_mid_3x3": bool(getattr(current_soup.net, "use_mid_3x3", True))},
    )
    artifacts["checkpoint"] = str(out_ckpt)
    print("Saved soup checkpoint to:", out_ckpt)

if SAVE_RESULTS_XLSX:
    out_xlsx = Path(f"{DIR}/{safe_tag}_results.xlsx")
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        history_df.to_excel(writer, sheet_name="greedy_history", index=False)
        current_fast_df.to_excel(writer, sheet_name="fast_eval_final", index=False)
        final_df.to_excel(writer, sheet_name="final_eval", index=False)
    artifacts["xlsx"] = str(out_xlsx)
    print("Saved results workbook to:", out_xlsx)

if SAVE_RESULTS_JSON:
    out_json = Path(f"{DIR}/{safe_tag}_recipe.json")
    payload = {
        "tag": safe_tag,
        "start_model": START_MODEL,
        "candidate_pool": CANDIDATE_POOL,
        "accepted_paths": accepted_paths,
        "metric_to_maximize": METRIC_TO_MAXIMIZE,
        "min_improvement": MIN_IMPROVEMENT,
        "fast_metric_final": float(current_fast_df.iloc[0][METRIC_TO_MAXIMIZE]),
        "final_metric": float(final_df.iloc[0][METRIC_TO_MAXIMIZE]),
        "fast_eval_row": current_fast_df.iloc[0].to_dict(),
        "final_eval_row": final_df.iloc[0].to_dict(),
        "history": history_df.to_dict(orient="records"),
    }
    out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    artifacts["json"] = str(out_json)
    print("Saved recipe JSON to:", out_json)

print("\nArtifacts:")
for k, v in artifacts.items():
    print(f"  {k}: {v}")


Saved soup checkpoint to: PPO_Models\PPO_SOUP__PPO_817.pt
Saved results workbook to: SOUP\PPO_SOUP__PPO_817_results.xlsx
Saved recipe JSON to: SOUP\PPO_SOUP__PPO_817_recipe.json

Artifacts:
  checkpoint: PPO_Models\PPO_SOUP__PPO_817.pt
  xlsx: SOUP\PPO_SOUP__PPO_817_results.xlsx
  json: SOUP\PPO_SOUP__PPO_817_recipe.json



## Notes

A few practical points:

- This notebook implements the exact greedy logic you described: **try candidate, keep only if improved, move on**.
- The soup is a **weight-space average** of accepted members.
- The final checkpoint is a single normal PPO/CNet192 checkpoint, not an inference ensemble.
- If the result is disappointing, that is useful information. It means the selected checkpoints do not live in a friendly shared basin, or your validation suite is not stressing the right blind spots.
- If this works, the accepted member list becomes the best teacher pool for **Track 2** routed distillation.
